In [ ]:
# Cell 1 - Convert per_day zips to CSV

import zipfile, time
from pathlib import Path
from io import BytesIO
import polars as pl

source_dir = Path(r"C:\Users\axels\CBOE_DATA_RAW\CBOE_DATA_2011_2022_ZIPPED")
dest_dir = Path(r"C:\Users\axels\CBOE_DATA_RAW_EXTRACTED")
dest_dir.mkdir(exist_ok=True)

zip_files = sorted(source_dir.glob("*.zip"))
print(f"Found {len(zip_files)} per-day zip files to convert\n")

with zipfile.ZipFile(zip_files[0], "r") as z:
    print("First zip contains:", z.namelist(), "\n")

EXCEL_EXT = (".xlsx", ".xls", ".xlsm")

t0 = time.time()
converted = 0
skipped = []

for i, zpath in enumerate(zip_files, 1):
    try:
        with zipfile.ZipFile(zpath, "r") as z:
            names = z.namelist()
            data_member, is_excel = None, False
            for n in names:
                low = n.lower()
                if low.endswith(".csv"):
                    data_member, is_excel = n, False
                    break
                if low.endswith(EXCEL_EXT):
                    data_member, is_excel = n, True
                    break
            if data_member is None:
                skipped.append((zpath.name, f"no CSV/Excel inside (found: {names})"))
                continue

            raw_bytes = z.read(data_member)
            out_path = dest_dir / f"{zpath.stem}.csv"

            if is_excel:
                df = pl.read_excel(BytesIO(raw_bytes))
                df.write_csv(out_path)
            else:
                out_path.write_bytes(raw_bytes)
        converted += 1
    except Exception as e:
        skipped.append((zpath.name, str(e)[:120]))

    if i % 100 == 0 or i == len(zip_files):
        elapsed = time.time() - t0
        remaining = (len(zip_files) - i) / (i / elapsed) / 60 if i else 0
        print(f"  {i}/{len(zip_files)} done ({elapsed/60:.1f} min elapsed, ~{remaining:.0f} min remaining)")

print(f"\nDone. {converted} files converted to CSV in {dest_dir}")
if skipped:
    print(f"{len(skipped)} file(s) had issues:")
    for name, reason in skipped[:10]:
        print(f"  - {name}: {reason}")

In [ ]:
# Cell 2 - Ingest CSVs into yearly Parquet files

from ingest_cboe import run_ingestion

summary = run_ingestion(
    raw_dir=r"C:\Users\axels\CBOE_DATA_RAW_EXTRACTED",
    out_dir=r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\cboe_parquet",
)

In [ ]:
# Cell 3 - Validate the Parquet output (row counts)

import polars as pl
from pathlib import Path

parquet_dir = Path(r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\cboe_parquet")
files = sorted(parquet_dir.glob("*.parquet"))
print(f"{len(files)} yearly files: {[f.stem for f in files]}\n")

total_rows = 0
for f in files:
    n = pl.scan_parquet(f).select(pl.len()).collect().item()
    total_rows += n
    print(f"{f.name}: {n:,} rows")

print(f"\nTotal rows across all years: {total_rows:,}")

In [ ]:
# Cell 4 - Disk usage check

import os, shutil
from pathlib import Path

def folder_size_gb(path):
    p = Path(path)
    if not p.exists():
        return None
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

locations = {
    "Per-day zip archives": r"C:\Users\axels\CBOE_DATA_RAW",
    "Extracted CSVs": r"C:\Users\axels\CBOE_DATA_RAW_EXTRACTED",
    "Final Parquet": r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\cboe_parquet",
}
for label, path in locations.items():
    size = folder_size_gb(path)
    print(f"{label}: {'not found' if size is None else f'{size:.2f} GB'} — {path}")

print(f"\nFree space: {shutil.disk_usage('C:/').free / 1e9:.1f} GB")

In [ ]:
# Cell 5 - Schema/quality/spot-check validation

import polars as pl
from pathlib import Path

parquet_dir = Path(r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\cboe_parquet")
extracted_dir = Path(r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\data\CBOE_DATA_RAW_EXTRACTED")
files = sorted(parquet_dir.glob("*.parquet"))

# 1. Schema consistency across all 12 years -- catches silent drift from diagonal_relaxed
print("=== 1. Schema consistency across years ===")
schemas = {f.stem: pl.scan_parquet(f).collect_schema() for f in files}
ref_name, ref_schema = next(iter(schemas.items()))
all_match = True
for name, schema in schemas.items():
    if schema != ref_schema:
        all_match = False
        diff_cols = set(schema.items()) ^ set(ref_schema.items())
        print(f"  MISMATCH in {name} vs {ref_name}: {diff_cols}")
print("  All schemas identical" if all_match else "  Differences found above -- investigate")

# 2. Per-year quality summary -- catches misfiled dates, unexpected nulls, bad value ranges
print("\n=== 2. Per-year data quality summary ===")
for f in files:
    stats = pl.scan_parquet(f).select(
        pl.len().alias("rows"),
        pl.col("quote_date").min().alias("min_date"),
        pl.col("quote_date").max().alias("max_date"),
        pl.col("quote_date").n_unique().alias("trading_days"),
        pl.col("strike_price").is_null().sum().alias("null_strikes"),
        pl.col("strike_price").min().alias("min_strike"),
        pl.col("strike_price").max().alias("max_strike"),
        pl.col("call_put_flag").is_null().sum().alias("null_cp_flag"),
    ).collect()
    print(f"  {f.stem}: {stats.to_dicts()[0]}")

# 3. Spot check: re-read ONE original CSV fresh and diff it against the parquet slice
#    for that exact date -- the strongest test, but only touches one file, not 2,862
print("\n=== 3. Spot check against a fresh source read ===")
sample_csv = extracted_dir / "C1OpenClose_2011-01-03.csv"
fresh = pl.read_csv(sample_csv, try_parse_dates=True)
from_parquet = pl.scan_parquet(parquet_dir / "cboe_openclose_2011.parquet").filter(
    pl.col("quote_date") == pl.date(2011, 1, 3)
).collect()

print(f"  fresh CSV rows: {fresh.height}, parquet rows for that date: {from_parquet.height}")
key_cols = ["option_symbol", "strike_price", "call_put_flag", "total_exchange_vol"]
fresh_sorted = fresh.select(key_cols).sort(key_cols)
pq_sorted = from_parquet.select(key_cols).sort(key_cols)
print("  values match exactly:", fresh_sorted.equals(pq_sorted))

In [ ]:
# Cell 6 - Path variables

from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "CBOE_DATA_RAW"
EXTRACTED_DIR = DATA_DIR / "CBOE_DATA_RAW_EXTRACTED"
PARQUET_DIR = PROJECT_ROOT / "cboe_parquet"